In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 1 — Installation des dépendances
!pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 114.3 MB/s eta 0:00:00


In [3]:
! pip install gradio

In [4]:
# Avec Reranker
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [5]:
# 2 — Imports & paramètres globaux
import os
import fitz
import faiss
import torch
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [6]:
# =========================
# PARAMÈTRES
# =========================
DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"





MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

CHUNK_SIZE = 250

OVERLAP = 30

TOP_K = 4

SIMILARITY_THRESHOLD = 0.35

MAX_CONTEXT_CHARS = 2000
MAX_NEW_TOKENS = 150


In [7]:
import os, json
from pathlib import Path

# =========================
# OUTILS
# =========================
def safe_get_text(obj):
    """
    Extrait du texte depuis différents formats JSON.
    - Si JSON concours structuré (keys concours_num + postes), on le "linéarise" en texte.
    - Sinon, fallback générique (dict/list/str).
    """
    if obj is None:
        return ""

    # --- CAS SPECIAL: JSON concours structuré ---
    if isinstance(obj, dict) and ("concours_num" in obj) and ("postes" in obj):
        lines = []

        # Métadonnées concours
        for k, label in [
            ("concours_label", "Concours"),
            ("concours_num", "Numéro"),
            ("bap", "BAP"),
            ("grade", "Grade"),
            ("emploi_type", "Emploi-type"),
            ("nb_postes", "Nombre de postes"),
            ("nb_postes_detectes", "Nombre de postes détectés"),
            ("source_file", "Source"),
        ]:
            v = obj.get(k, None)
            if v is not None and str(v).strip() != "":
                lines.append(f"{label} : {v}")

        # Détails postes
        postes = obj.get("postes", [])
        if isinstance(postes, list) and postes:
            for i, p in enumerate(postes, start=1):
                if not isinstance(p, dict):
                    continue
                lines.append(f"\nPOSTE {i} :")
                # On prend toutes les infos du poste, même si les clés varient
                for pk, pv in p.items():
                    if pv is None:
                        continue
                    # listes -> concat
                    if isinstance(pv, list):
                        pv = " ; ".join(str(x) for x in pv if str(x).strip())
                    # dict -> string simple
                    elif isinstance(pv, dict):
                        pv = " ; ".join(f"{a}={b}" for a, b in pv.items() if str(b).strip())
                    else:
                        pv = str(pv)

                    pv = pv.strip()
                    if pv:
                        lines.append(f"- {pk} : {pv}")

        return "\n".join(lines).strip()

    # --- FALLBACK GENERIQUE ---
    if isinstance(obj, str):
        return obj.strip()

    if isinstance(obj, list):
        parts = []
        for it in obj:
            t = safe_get_text(it)
            if t:
                parts.append(t)
        return "\n".join(parts).strip()

    if isinstance(obj, dict):
        # si jamais un JSON "texte" existe
        for k in ["text", "content", "clean_text", "raw_text", "body", "page_content"]:
            if k in obj and isinstance(obj[k], str) and obj[k].strip():
                return obj[k].strip()

        parts = []
        for v in obj.values():
            t = safe_get_text(v)
            if t:
                parts.append(t)
        return "\n".join(parts).strip()

    return ""


def chunk_text(text, chunk_size=250, overlap=30):
    """
    Chunk par mots, avec overlap.
    """
    words = text.split()
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size]).strip()
        if chunk:
            yield chunk


# =========================
# 1) CHECK DOSSIER
# =========================
print("DATA_DIR exists:", os.path.isdir(DATA_DIR))
print("Exemples fichiers:", os.listdir(DATA_DIR)[:10])

# =========================
# 2) CHARGEMENT JSON
# =========================
json_files = sorted([p for p in Path(DATA_DIR).glob("*.json")])
print("Nb fichiers JSON:", len(json_files))

documents = []
skipped = 0

for fp in json_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            obj = json.load(f)

        full_text = safe_get_text(obj)
        if not full_text:
            skipped += 1
            continue

        # Métadonnées source
        source = fp.name

        # Déduire la page depuis le nom du fichier (ex: page_009.json -> 9)
        page = "N/A"
        if fp.stem.startswith("page_"):
            try:
                page = int(fp.stem.split("_")[1])
            except:
                page = "N/A"

        # Chunk
        for chunk in chunk_text(full_text, CHUNK_SIZE, OVERLAP):
            documents.append({
                "text": chunk,
                "source": source,
                "page": page
            })

    except Exception as e:
        skipped += 1
        print(f"⚠️ Erreur fichier {fp.name}: {e}")


print(f"✅ Chunks créés: {len(documents)}")
print(f"⚠️ Fichiers ignorés/erreurs: {skipped}")

# =========================
# 3) APERÇU
# =========================
if documents:
    print("\n--- Exemple chunk ---")
    print("SOURCE:", documents[0]["source"])
    print(documents[0]["text"][:800])


DATA_DIR exists: True
Exemples fichiers: ['page_009.json', 'page_026.json', 'page_048.json', 'page_102.json', 'page_049.json', 'page_072.json', 'page_065.json', 'page_056.json', 'page_080.json', 'page_051.json']
Nb fichiers JSON: 127
✅ Chunks créés: 625
⚠️ Fichiers ignorés/erreurs: 0

--- Exemple chunk ---
SOURCE: guide_candidat_2025.json
CNRS – Guide candidat(e) 2025 (IT) Guide candidat 2025.pdf CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 Direction de la publication : Antoine Petit Direction de la rédaction : Hélène Maury Direction adjointe de la rédaction : Christiane Ename – Laetitia Navarro -Service recrutement et intégration (SeRI) Autrices : Dominique Marx - Emilie Faure - Nathalie Nioucel Mai 2025 5 - 6 Pourquoi candidater ? 7 - 8 Le choix des concours 9 - 10 L’inscription 11 Comment concourir ? 12 Les conditions pour concourir 13 - 14 Le déroulement des concours 15-16 Les épreuves 17 La publication des résultat

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# 3 — Détection GPU automatique
def can_use_gpu(min_free_gb=4):
    if not torch.cuda.is_available():
        return False
    free, total = torch.cuda.mem_get_info()
    return free / (1024**3) >= min_free_gb

USE_GPU = can_use_gpu()
print(f"🔍 Mode sélectionné : {'GPU' if USE_GPU else 'CPU'}")


🔍 Mode sélectionné : GPU


In [10]:
import json
from pathlib import Path

def safe_get_text(obj):
    if isinstance(obj, dict):
        for k in ["text", "content", "clean_text", "body"]:
            if k in obj and isinstance(obj[k], str):
                return obj[k].strip()
        for v in obj.values():
            t = safe_get_text(v)
            if t:
                return t
    if isinstance(obj, list):
        return "\n".join(filter(None, (safe_get_text(x) for x in obj)))
    if isinstance(obj, str):
        return obj.strip()
    return ""


def chunk_text(text):
    words = text.split()
    step = max(1, CHUNK_SIZE - OVERLAP)
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + CHUNK_SIZE])


documents = []

json_files = list(Path(DATA_DIR).glob("*.json"))
print("📁 Fichiers JSON trouvés :", len(json_files))

for fp in json_files:
    with open(fp, "r", encoding="utf-8") as f:
        obj = json.load(f)

    full_text = safe_get_text(obj)
    if not full_text.strip():
        continue

    for chunk in chunk_text(full_text):
        documents.append({
            "text": chunk,
            "source": fp.name
        })

print(f"📄 Documents indexés : {len(documents)} chunks")


📁 Fichiers JSON trouvés : 127
📄 Documents indexés : 127 chunks


In [11]:
# 5 — Embeddings & FAISS
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

assert len(documents) > 0, "documents est vide — vérifie le chargement/chunking avant FAISS."

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if USE_GPU else "cpu"
)

texts = [d["text"] for d in documents]

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,   # => cosine similarity si IndexFlatIP
    batch_size=16,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")
embeddings = np.ascontiguousarray(embeddings)  # FAISS aime le contigu

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"✅ FAISS prêt — {index.ntotal} vecteurs, dim={dim}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ FAISS prêt — 127 vecteurs, dim=384


In [12]:
# 6 — Retrieval avec Reranker
import numpy as np

def retrieve(question):
    # 1) Retrieval large
    q_emb = embedder.encode([question], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")
    q_emb = np.ascontiguousarray(q_emb)

    scores, indices = index.search(q_emb, TOP_K * 3)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        # seuil "soft" (optionnel) : garde-le bas pour ne pas perdre des bons passages
        if score >= SIMILARITY_THRESHOLD:
            candidates.append({**documents[idx], "retrieval_score": float(score)})

    if not candidates:
        return []

    # 2) Reranking précis
    pairs = [(question, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)

    reranked = sorted(
        zip(rerank_scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )

    # 3) On garde les meilleurs
    results = []
    for s, c in reranked[:TOP_K]:
        c2 = dict(c)
        c2["rerank_score"] = float(s)
        results.append(c2)

    return results


In [13]:
# 7 — Chargement du modèle Mistral (fix accelerate/pipeline)
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if USE_GPU:
    # GPU (accelerate device_map auto)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

else:
    # CPU (pas accelerate)
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    torch.set_num_threads(4)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=-1,  # optionnel, mais OK sur CPU
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

print("✅ Modèle + pipeline prêts :", MODEL_NAME, "| GPU" if USE_GPU else "| CPU")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Modèle + pipeline prêts : mistralai/Mistral-7B-Instruct-v0.2 | GPU


In [14]:
# 8 — Prompt CNRS STRICT (anti-hallucination)
def build_prompt(question, contexts):
    def fmt_source(c):
        page = c.get("page", None)
        if page is None or page == "N/A":
            return f"{c.get('source', 'inconnu')}"
        return f"{c.get('source', 'inconnu')} | page {page}"

    # Contexte (on limite chaque extrait + on limite le total)
    blocks = []
    total_chars = 0

    for c in contexts:
        excerpt = (c.get("text", "") or "").strip()
        if not excerpt:
            continue

        excerpt = excerpt[:MAX_CONTEXT_CHARS]
        block = f"- {fmt_source(c)}\n  {excerpt}"

        if total_chars + len(block) > MAX_CONTEXT_CHARS * max(1, TOP_K):
            break

        blocks.append(block)
        total_chars += len(block)

    sources_block = "\n".join(blocks) if blocks else "- (aucun contexte)"

    return f"""<s>[INST] Tu es un agent officiel d'information sur les concours ingénieur du CNRS.

RÈGLES ABSOLUES :
- Tu utilises EXCLUSIVEMENT les sources ci-dessous.
- Tu ne déduis rien.
- Tu ne complètes rien.
- Tu ne poses pas de nouvelle question.
- Tu ne réponds qu'UNE SEULE FOIS.
- Tu réponds uniquement en français.

FORMAT DE SORTIE OBLIGATOIRE :

RÉPONSE :
<réponse factuelle>

SOURCES :
- <fichier> | page <numéro>

SI l'information n'est PAS clairement présente, répond EXACTEMENT :

RÉPONSE :
Je ne dispose pas de cette information dans les documents de référence.

SOURCES :
Aucune

SOURCES DISPONIBLES :
{sources_block}

QUESTION :
{question} [/INST]
"""


In [15]:
# Bloquer / nettoyer la sortie du modèle
def clean_output(text):
    stop_markers = [
        "\nQUESTION :",
        "\n❓",
        "\n[INST]",
        "\nSOURCES DISPONIBLES",
        "\nSOURCES :",
        "\nSOURCE :",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return text.strip()


In [16]:
# 9 — Fonction answer() finale
REFUS = "Je ne dispose pas de cette information dans les documents de référence."

def answer(question):
    contexts = retrieve(question)

    # Si rien trouvé
    if not contexts:
        return f"RÉPONSE :\n{REFUS}\n\nSOURCES :\nAucune"

    prompt = build_prompt(question, contexts)

    # Génération
    gen = pipe(prompt)[0]["generated_text"]
    output = clean_output(gen)

    # Si le modèle refuse (ou ne respecte pas le format), on force le refus
    if (REFUS in output) or ("RÉPONSE :" not in output):
        return f"RÉPONSE :\n{REFUS}\n\nSOURCES :\nAucune"

    return output


In [17]:
# 10 — Mode interactif (démo)
print("\n🤖 Agent CNRS prêt. Tape 'quitter' pour quitter.\n")

while True:
    q = input("❓ Question : ").strip()

    if not q:
        continue

    if q.lower() in {"quitter", "quit", "exit"}:
        break

    print("\n" + answer(q))
    print("\n" + "-" * 60)



🤖 Agent CNRS prêt. Tape 'quitter' pour quitter.

❓ Question : Quels sont les avantages à travailler comme ingénieur au CNRS en termes de carrière et de conditions de travail ?

RÉPONSE :
Les avantages en matière de carrière et de conditions de travail pour travailler comme ingénieur au CNRS sont détaillés dans le document "CNRS Carrières – Vos avantages" (page 2 et suivantes).

------------------------------------------------------------
❓ Question : exit


In [ ]:
import gradio as gr
from PIL import Image
import time
from datetime import datetime

# ============================================================
# ======================= LOGIQUE CHAT =======================
# ============================================================

def chat_fn(message, history):
    """Traite un message et retourne l'historique mis à jour"""
    if not message or not message.strip():
        return history, ""

    # Appel à la fonction answer (à définir ailleurs dans votre code)
    response = answer(message)
    history.append((message, response))
    return history, ""

def quick_question(question, history):
    """Traite une question rapide depuis les boutons"""
    response = answer(question)
    history.append((question, response))
    return history

def handle_example_click(example_text, history):
    """Gère le clic sur un exemple"""
    # Retire les emojis du début
    clean_question = example_text.split(" ", 1)[1] if " " in example_text else example_text
    return quick_question(clean_question, history)

# ============================================================
# ===================== MESSAGES & CONFIG ====================
# ============================================================

WELCOME = """<div style='padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 20px; color: white; box-shadow: 0 10px 40px rgba(102, 126, 234, 0.25);'>
<div style='text-align: center; margin-bottom: 20px;'>
<h2 style='margin: 0 0 8px 0; font-size: 2em; font-weight: 800;'>👋 Bienvenue sur l'Agent CNRS Intelligence</h2>
<p style='margin: 0; font-size: 1.15em; opacity: 0.95; font-weight: 500;'>
Votre assistant IA expert pour les concours et carrières d'ingénieur
</p>
</div>

<div style='background: rgba(255,255,255,0.15); padding: 20px; border-radius: 12px; margin: 20px 0; backdrop-filter: blur(10px);'>
<p style='margin: 0 0 15px 0; font-size: 1.05em;'><strong>🎯 Je suis spécialisé dans :</strong></p>
<div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 12px; margin-top: 12px;'>
<div style='background: rgba(255,255,255,0.1); padding: 12px; border-radius: 8px; border-left: 3px solid #ffd700;'>
<strong>🔍 Orientation personnalisée</strong><br/>
<span style='font-size: 0.9em; opacity: 0.9;'>Identifier les concours adaptés à votre profil et compétences</span>
</div>
<div style='background: rgba(255,255,255,0.1); padding: 12px; border-radius: 8px; border-left: 3px solid #4ade80;'>
<strong>📊 Informations complètes</strong><br/>
<span style='font-size: 0.9em; opacity: 0.9;'>Grades, salaires, avantages et évolution de carrière</span>
</div>
<div style='background: rgba(255,255,255,0.1); padding: 12px; border-radius: 8px; border-left: 3px solid #60a5fa;'>
<strong>📝 Guide d'inscription</strong><br/>
<span style='font-size: 0.9em; opacity: 0.9;'>Procédures, documents requis, dates et échéances</span>
</div>
<div style='background: rgba(255,255,255,0.1); padding: 12px; border-radius: 8px; border-left: 3px solid #f472b6;'>
<strong>✅ Réponses certifiées</strong><br/>
<span style='font-size: 0.9em; opacity: 0.9;'>Basées sur 122 concours officiels + 4 documents de référence</span>
</div>
</div>
</div>

<div style='background: rgba(255,215,0,0.15); padding: 15px; border-radius: 10px; border: 2px solid rgba(255,215,0,0.4); margin-top: 15px;'>
<strong>⚡ Engagement Qualité :</strong> Si une information n'est pas dans mes documents officiels,
je vous le dirai clairement. <strong>Zéro approximation, 100% fiabilité.</strong>
</div>

<p style='text-align: center; margin: 20px 0 0 0; font-size: 0.95em; opacity: 0.9;'>
💬 <em>Commencez par poser votre question ou explorez les suggestions ci-dessous</em>
</p>
</div>"""

EXAMPLES = [
    ("🔍", "Quels concours pour un profil en bioinformatique ?", "#3b82f6"),
    ("💰", "Grille de rémunération des ingénieurs de recherche", "#10b981"),
    ("📅", "Déroulement complet d'un concours CNRS", "#8b5cf6"),
    ("🎯", "Différence entre ingénieur d'étude et de recherche", "#f59e0b"),
    ("🏛️", "Avantages et conditions de travail au CNRS", "#ec4899"),
    ("📄", "Documents nécessaires pour candidater", "#06b6d4"),
    ("🔬", "Concours en sciences du vivant disponibles", "#14b8a6"),
    ("⏰", "Dates limites d'inscription 2026", "#ef4444")
]

# ============================================================
# ======================= IMAGE LOGO =========================
# ============================================================

LOGO_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Celia Marmouget/logo_cnrs.png"
try:
    logo_cnrs = Image.open(LOGO_PATH)
except FileNotFoundError:
    logo_cnrs = None
    print(f"⚠️ Logo non trouvé à {LOGO_PATH}")

# ============================================================
# ===================== THÈME PERSONNALISÉ ===================
# ============================================================

custom_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="purple",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter"),
    font_mono=gr.themes.GoogleFont("JetBrains Mono")
).set(
    body_background_fill="#f8f9fa",
    body_background_fill_dark="#1a1a2e",
    button_primary_background_fill="#0055A4",
    button_primary_background_fill_hover="#003D75",
    button_primary_text_color="white",
    button_secondary_background_fill="white",
    button_secondary_background_fill_hover="#f0f2f5",
    input_background_fill="white",
    input_border_color="#e1e4e8"
)

# ============================================================
# ======================= CSS PREMIUM ========================
# ============================================================

custom_css = """
/* ===== IMPORTS & VARIABLES ===== */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
    --primary-color: #0055A4;
    --primary-dark: #003D75;
    --gradient-purple: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    --gradient-blue: linear-gradient(135deg, #0055A4 0%, #003D75 100%);
    --shadow-sm: 0 2px 8px rgba(0,0,0,0.04);
    --shadow-md: 0 4px 16px rgba(0,0,0,0.08);
    --shadow-lg: 0 8px 32px rgba(0,0,0,0.12);
    --shadow-xl: 0 12px 48px rgba(0,0,0,0.15);
    --transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
}

/* ===== ANIMATIONS ===== */
@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(30px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes fadeIn {
    from { opacity: 0; }
    to { opacity: 1; }
}

@keyframes slideInLeft {
    from { opacity: 0; transform: translateX(-40px); }
    to { opacity: 1; transform: translateX(0); }
}

@keyframes slideInRight {
    from { opacity: 0; transform: translateX(40px); }
    to { opacity: 1; transform: translateX(0); }
}

@keyframes scaleIn {
    from { opacity: 0; transform: scale(0.9); }
    to { opacity: 1; transform: scale(1); }
}

@keyframes pulse {
    0%, 100% { transform: scale(1); opacity: 1; }
    50% { transform: scale(1.05); opacity: 0.9; }
}

@keyframes shimmer {
    0% { background-position: -1000px 0; }
    100% { background-position: 1000px 0; }
}

@keyframes float {
    0%, 100% { transform: translateY(0px); }
    50% { transform: translateY(-10px); }
}

/* ===== CONTENEUR PRINCIPAL ===== */
.gradio-container {
    max-width: 1400px !important;
    margin: 0 auto !important;
    animation: fadeIn 0.8s ease-out;
}

#component-0 {
    max-width: 1400px !important;
    margin: 0 auto !important;
}

/* ===== HEADER SPECTACULAIRE ===== */
.header-section {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 50px 40px;
    border-radius: 24px;
    margin-bottom: 35px;
    box-shadow: 0 20px 60px rgba(102, 126, 234, 0.35);
    position: relative;
    overflow: hidden;
    animation: fadeInUp 1s ease-out;
}

.header-section::before {
    content: '';
    position: absolute;
    top: -50%;
    right: -20%;
    width: 150%;
    height: 200%;
    background: radial-gradient(circle, rgba(255,255,255,0.15) 0%, transparent 70%);
    animation: pulse 6s ease-in-out infinite;
}

.header-section::after {
    content: '';
    position: absolute;
    bottom: 0;
    left: 0;
    right: 0;
    height: 3px;
    background: linear-gradient(90deg, #ffd700, #4ade80, #60a5fa, #f472b6);
    background-size: 300% 100%;
    animation: shimmer 3s linear infinite;
}

.logo-container {
    display: flex;
    justify-content: center;
    margin-bottom: 25px;
    animation: float 3s ease-in-out infinite;
}

.logo-container img {
    filter: drop-shadow(0 10px 25px rgba(0,0,0,0.3));
    transition: transform 0.4s cubic-bezier(0.34, 1.56, 0.64, 1);
}

.logo-container img:hover {
    transform: scale(1.1) rotate(5deg);
}

.title-main {
    color: white !important;
    font-size: 3em !important;
    font-weight: 800 !important;
    margin: 0 0 12px 0 !important;
    text-align: center;
    text-shadow: 0 4px 20px rgba(0,0,0,0.3);
    letter-spacing: -1px;
    line-height: 1.2;
}

.subtitle-main {
    color: rgba(255,255,255,0.95) !important;
    font-size: 1.3em !important;
    text-align: center;
    font-weight: 500;
    margin: 0;
    text-shadow: 0 2px 10px rgba(0,0,0,0.2);
}

/* ===== BADGES STATISTIQUES ===== */
.stats-container {
    display: flex;
    justify-content: center;
    gap: 15px;
    flex-wrap: wrap;
    padding: 25px 20px;
    animation: fadeInUp 1.2s ease-out;
}

.stats-badge {
    display: inline-flex;
    align-items: center;
    gap: 10px;
    background: white;
    padding: 14px 24px;
    border-radius: 50px;
    font-size: 1em;
    font-weight: 700;
    box-shadow: var(--shadow-md);
    transition: var(--transition);
    border: 2px solid transparent;
}

.stats-badge:hover {
    transform: translateY(-4px);
    box-shadow: var(--shadow-lg);
    border-color: var(--primary-color);
}

/* ===== SECTIONS ===== */
.section-card {
    background: white;
    padding: 30px;
    border-radius: 20px;
    box-shadow: var(--shadow-md);
    margin-bottom: 30px;
    border: 1px solid rgba(0,0,0,0.05);
    transition: var(--transition);
    animation: fadeInUp 1.4s ease-out;
}

.section-card:hover {
    box-shadow: var(--shadow-lg);
    transform: translateY(-2px);
}

.section-title {
    font-size: 1.6em;
    font-weight: 800;
    color: #1a202c;
    margin: 0 0 25px 0;
    display: flex;
    align-items: center;
    gap: 12px;
    padding-bottom: 15px;
    border-bottom: 3px solid #e2e8f0;
}

.section-title::before {
    content: '';
    width: 5px;
    height: 30px;
    background: var(--gradient-blue);
    border-radius: 10px;
}

/* ===== BOUTONS QUESTIONS RAPIDES ===== */
.quick-btn {
    min-height: 70px !important;
    font-size: 1.05em !important;
    font-weight: 700 !important;
    border: 2px solid #e2e8f0 !important;
    border-radius: 16px !important;
    transition: all 0.4s cubic-bezier(0.34, 1.56, 0.64, 1) !important;
    background: linear-gradient(135deg, white 0%, #f8f9fa 100%) !important;
    box-shadow: var(--shadow-sm) !important;
    position: relative !important;
    overflow: hidden !important;
}

.quick-btn::before {
    content: '';
    position: absolute;
    top: 0;
    left: -100%;
    width: 100%;
    height: 100%;
    background: linear-gradient(90deg, transparent, rgba(255,255,255,0.3), transparent);
    transition: left 0.5s;
}

.quick-btn:hover::before {
    left: 100%;
}

.quick-btn:hover {
    transform: translateY(-6px) scale(1.02) !important;
    box-shadow: 0 12px 35px rgba(0, 85, 164, 0.25) !important;
    border-color: var(--primary-color) !important;
    background: var(--gradient-blue) !important;
    color: white !important;
}

/* ===== EXEMPLES DE QUESTIONS ===== */
.examples-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
    gap: 12px;
    margin-top: 20px;
}

.example-card {
    background: white;
    padding: 16px 20px;
    border-radius: 12px;
    cursor: pointer;
    transition: var(--transition);
    border: 2px solid #f0f2f5;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 0.98em;
    font-weight: 600;
    box-shadow: var(--shadow-sm);
}

.example-card:hover {
    background: var(--gradient-blue);
    color: white;
    transform: translateX(10px) scale(1.02);
    box-shadow: var(--shadow-md);
    border-color: var(--primary-color);
}

.example-emoji {
    font-size: 1.8em;
    min-width: 40px;
    text-align: center;
}

/* ===== CHAT ===== */
.chat-section {
    animation: fadeInUp 1.6s ease-out;
}

.chatbot-container {
    border: 2px solid #e2e8f0 !important;
    border-radius: 16px !important;
    box-shadow: var(--shadow-md) !important;
    overflow: hidden !important;
    background: linear-gradient(to bottom, #ffffff 0%, #f8f9fa 100%) !important;
}

/* ===== INPUT ZONE ===== */
.input-zone {
    background: white;
    padding: 25px;
    border-radius: 16px;
    box-shadow: 0 -6px 30px rgba(0,0,0,0.06);
    margin-top: 20px;
    border: 2px solid #f0f2f5;
}

.message-input {
    border: 2px solid #e2e8f0 !important;
    border-radius: 14px !important;
    padding: 18px 20px !important;
    font-size: 1.05em !important;
    transition: var(--transition) !important;
    background: #fafbfc !important;
    font-weight: 500 !important;
}

.message-input:focus {
    border-color: var(--primary-color) !important;
    box-shadow: 0 0 0 4px rgba(0, 85, 164, 0.1) !important;
    background: white !important;
    transform: translateY(-2px);
}

.send-button {
    background: var(--gradient-blue) !important;
    border: none !important;
    border-radius: 14px !important;
    padding: 18px 36px !important;
    font-weight: 800 !important;
    font-size: 1.1em !important;
    transition: var(--transition) !important;
    box-shadow: 0 6px 20px rgba(0, 85, 164, 0.35) !important;
    position: relative !important;
    overflow: hidden !important;
}

.send-button::after {
    content: '→';
    position: absolute;
    right: 20px;
    opacity: 0;
    transition: var(--transition);
}

.send-button:hover::after {
    opacity: 1;
    right: 15px;
}

.send-button:hover {
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 10px 30px rgba(0, 85, 164, 0.45) !important;
}

.send-button:active {
    transform: translateY(-1px) scale(0.98) !important;
}

/* ===== BOUTONS SECONDAIRES ===== */
.secondary-btn {
    border-radius: 10px !important;
    padding: 10px 20px !important;
    font-weight: 600 !important;
    transition: var(--transition) !important;
    border: 2px solid #e2e8f0 !important;
}

.secondary-btn:hover {
    border-color: var(--primary-color) !important;
    background: #f8f9fa !important;
    transform: translateY(-2px) !important;
}

/* ===== FOOTER ===== */
.footer-info {
    background: linear-gradient(135deg, #f7fafc 0%, #edf2f7 100%);
    padding: 30px;
    border-radius: 16px;
    margin-top: 30px;
    border-left: 5px solid var(--primary-color);
    box-shadow: var(--shadow-md);
    animation: fadeInUp 1.8s ease-out;
}

.footer-info h3 {
    color: var(--primary-color);
    font-size: 1.4em;
    margin: 0 0 15px 0;
    font-weight: 800;
}

.footer-info p {
    color: #4a5568;
    line-height: 1.8;
    margin: 10px 0;
    font-size: 1.02em;
}

.footer-info a {
    color: var(--primary-color);
    font-weight: 700;
    text-decoration: none;
    border-bottom: 2px solid transparent;
    transition: var(--transition);
    padding-bottom: 2px;
}

.footer-info a:hover {
    border-bottom-color: var(--primary-color);
    color: var(--primary-dark);
}

.contact-badge {
    display: inline-block;
    background: white;
    padding: 8px 16px;
    border-radius: 20px;
    margin: 5px;
    font-weight: 600;
    box-shadow: var(--shadow-sm);
    font-size: 0.95em;
}

/* ===== RESPONSIVE ===== */
@media (max-width: 768px) {
    .title-main { font-size: 2.2em !important; }
    .header-section { padding: 30px 20px; }
    .quick-btn { min-height: 60px !important; font-size: 0.95em !important; }
    .stats-container { gap: 10px; }
    .stats-badge { padding: 10px 16px; font-size: 0.9em; }
    .examples-grid { grid-template-columns: 1fr; }
    .section-title { font-size: 1.3em; }
}

/* ===== SCROLLBAR PERSONNALISÉE ===== */
::-webkit-scrollbar {
    width: 10px;
    height: 10px;
}

::-webkit-scrollbar-track {
    background: #f1f1f1;
    border-radius: 10px;
}

::-webkit-scrollbar-thumb {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    border-radius: 10px;
}

::-webkit-scrollbar-thumb:hover {
    background: var(--gradient-blue);
}
"""

# ============================================================
# ======================= INTERFACE ==========================
# ============================================================

with gr.Blocks(theme=custom_theme, css=custom_css, title="🔬 Agent CNRS Intelligence | Concours & Recrutements") as demo:

    # ===== HEADER SPECTACULAIRE =====
    with gr.Column(elem_classes="header-section"):
        if logo_cnrs:
            gr.Image(
                value=logo_cnrs,
                show_label=False,
                height=140,
                width=140,
                interactive=False,
                elem_classes="logo-container"
            )

        gr.Markdown(
            """
            <h1 class="title-main">🔬 Agent CNRS Intelligence</h1>
            <p class="subtitle-main">Assistant IA Expert · Concours & Carrières d'Ingénieur · 100% Officiel</p>
            """
        )

    # ===== BADGES STATISTIQUES ANIMÉS =====
    gr.HTML(
        """
        <div class="stats-container">
            <div class="stats-badge" style="border-left: 4px solid #3b82f6;">
                <span style="font-size: 1.5em;">📊</span>
                <span>170+ Concours/an</span>
            </div>
            <div class="stats-badge" style="border-left: 4px solid #10b981;">
                <span style="font-size: 1.5em;">🎯</span>
                <span>Réponses Sourcées</span>
            </div>
            <div class="stats-badge" style="border-left: 4px solid #f59e0b;">
                <span style="font-size: 1.5em;">⚡</span>
                <span>Instantané</span>
            </div>
            <div class="stats-badge" style="border-left: 4px solid #8b5cf6;">
                <span style="font-size: 1.5em;">✅</span>
                <span>100% Officiel</span>
            </div>
        </div>
        """
    )

    # ===== QUESTIONS RAPIDES =====
    with gr.Column(elem_classes="section-card"):
        gr.Markdown('<h2 class="section-title">⚡ Questions Rapides</h2>')

        with gr.Row(equal_height=True):
            q1 = gr.Button("📘 Types de concours & domaines scientifiques", elem_classes="quick-btn", scale=1)
            q2 = gr.Button("📝 Guide complet d'inscription", elem_classes="quick-btn", scale=1)

        with gr.Row(equal_height=True):
            q3 = gr.Button("📅 Calendrier complet 2026 & échéances", elem_classes="quick-btn", scale=1)
            q4 = gr.Button("🎓 Carrières, grades & rémunération", elem_classes="quick-btn", scale=1)

    # ===== EXEMPLES DE QUESTIONS =====
    with gr.Column(elem_classes="section-card"):
        gr.Markdown('<h2 class="section-title">💡 Exemples de Questions</h2>')

        examples_html = '<div class="examples-grid">'
        for emoji, text, color in EXAMPLES:
            examples_html += f'''
            <div class="example-card" style="border-left: 4px solid {color};" onclick="document.querySelector('.message-input textarea').value = '{text}'; document.querySelector('.message-input textarea').dispatchEvent(new Event('input', {{ bubbles: true }}));">
                <span class="example-emoji">{emoji}</span>
                <span>{text}</span>
            </div>
            '''
        examples_html += '</div>'

        gr.HTML(examples_html)

    # ===== CHAT =====
    with gr.Column(elem_classes="section-card chat-section"):
        gr.Markdown('<h2 class="section-title">💬 Conversation avec l\'Agent IA</h2>')

        chatbot = gr.Chatbot(
            value=[(None, WELCOME)],
            height=520,
            show_label=False,
            avatar_images=(None, "🤖"),
            elem_classes="chatbot-container",
            show_copy_button=True,
            type="tuples"
        )

        with gr.Column(elem_classes="input-zone"):
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="💬 Posez votre question ici... (ex: Quels concours pour un profil en data science avec un doctorat ?)",
                    show_label=False,
                    scale=8,
                    lines=2,
                    max_lines=5,
                    elem_classes="message-input"
                )
                submit_btn = gr.Button("Envoyer 🚀", scale=2, variant="primary", elem_classes="send-button")

            with gr.Row():
                clear_btn = gr.Button("🗑️ Nouvelle conversation", size="sm", variant="secondary", elem_classes="secondary-btn")
                gr.Button("💾 Exporter (PDF)", size="sm", variant="secondary", elem_classes="secondary-btn")
                gr.Button("📋 Copier tout", size="sm", variant="secondary", elem_classes="secondary-btn")

    # ===== FOOTER ENRICHI =====
    with gr.Column(elem_classes="footer-info"):
        gr.Markdown(
            """
            <h3>ℹ️ À propos de cet Agent IA</h3>
            <p>
            <strong>🎓 Contexte :</strong> Développé dans le cadre du <strong>Hackathon SID 2025</strong>
            en collaboration avec le <strong>CNRS Délégation Occitanie-Ouest</strong>.
            Cet agent conversationnel exploite l'intelligence artificielle pour vous guider dans votre recherche de carrière au CNRS.
            </p>
            <p>
            <strong>📚 Base de connaissances :</strong> <span style="background: #e0f2fe; padding: 4px 12px; border-radius: 6px; font-weight: 600;">122 pages HTML de concours officiels</span> +
            <span style="background: #fef3c7; padding: 4px 12px; border-radius: 6px; font-weight: 600;">4 documents de référence CNRS</span> +
            <span style="background: #ddd6fe; padding: 4px 12px; border-radius: 6px; font-weight: 600;">Données 2025-2026</span>
            </p>
            <p>
            <strong>🔒 Engagement éthique :</strong> Toutes les réponses sont basées <strong>exclusivement</strong> sur des documents officiels.
            Notre priorité : <strong>fiabilité, transparence et précision</strong>. Si une information n'est pas disponible, nous vous l'indiquerons clairement.
            </p>
            <p style="margin-top: 18px; padding-top: 18px; border-top: 2px solid #e2e8f0;">
            <strong>🌐 Ressources officielles :</strong><br/>
            <a href="https://www.dgdr.cnrs.fr/drhchercheurs/concoursch/default-fr.htm" target="_blank">🔗 Portail des Concours CNRS</a> •
            <a href="https://www.cnrs.fr/fr/page/travailler-au-cnrs" target="_blank">🔗 Carrières au CNRS</a> •
            <a href="https://www.cnrs.fr" target="_blank">🔗 Site Officiel CNRS</a>
            </p>
            <p style="margin-top: 20px; padding: 20px; background: white; border-radius: 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.05);">
            <strong>👥 Contacts Référents Hackathon :</strong><br/>
            <span class="contact-badge">👨‍🔬 Laurent RISSER · Ingénieur de Recherche HDR · lrisser@math.univ-toulouse.fr</span>
            <span class="contact-badge">👨‍💼 Stéphane LEBLANC · stephane.leblanc@cnrs.fr</span>
            </p>
            <p style="text-align: center; margin-top: 20px; padding-top: 15px; border-top: 1px dashed #cbd5e0; color: #718096; font-size: 0.92em;">
            <em>🚀 Hackathon SID 2025 · Intelligence Artificielle au service de la Recherche Publique</em>
            </p>
            """
        )

    # ===== INTERACTIONS OPTIMISÉES =====

    # Soumission du message (Enter ou bouton)
    submit_msg = msg.submit(chat_fn, [msg, chatbot], [chatbot, msg], queue=False)
    submit_btn_click = submit_btn.click(chat_fn, [msg, chatbot], [chatbot, msg], queue=False)

    # Questions rapides avec formulations optimales
    q1.click(
        lambda h: quick_question(
            "Peux-tu me présenter tous les types de concours proposés par le CNRS et les principaux domaines scientifiques couverts ?",
            h
        ),
        [chatbot],
        chatbot,
        queue=False
    )

    q2.click(
        lambda h: quick_question(
            "Explique-moi étape par étape comment m'inscrire à un concours CNRS : documents nécessaires, procédure en ligne, et conseils pratiques.",
            h
        ),
        [chatbot],
        chatbot,
        queue=False
    )

    q3.click(
        lambda h: quick_question(
            "Quel est le calendrier complet des concours CNRS pour 2026 ? Dates d'ouverture, de clôture, d'auditions et de résultats.",
            h
        ),
        [chatbot],
        chatbot,
        queue=False
    )

    q4.click(
        lambda h: quick_question(
            "Détaille-moi les différents grades d'ingénieur au CNRS (IR, IE...), leurs grilles de rémunération, avantages sociaux et perspectives d'évolution de carrière.",
            h
        ),
        [chatbot],
        chatbot,
        queue=False
    )

    # Réinitialisation de la conversation
    clear_btn.click(
        lambda: [(None, WELCOME)],
        None,
        chatbot,
        queue=False
    )

# ============================================================
# ======================= LANCEMENT ==========================
# ============================================================

if __name__ == "__main__":
    demo.launch(
        share=True,
        server_name="0.0.0.0",
        show_error=True,
        favicon_path=LOGO_PATH if logo_cnrs else None,
        show_api=False
    )